In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

print(cwd)

for candidate in [cwd, *cwd.parents]:
    # print(f"Looking for {marker} in {candidate}")
    if (candidate / marker).exists():
        #print("Project root :", candidate)
        PROJECT_ROOT = str(candidate)
        #print("Project root :", PROJECT_ROOT)
        break


if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
dbutils.widgets.dropdown(
    "environment", "DEV", ["DEV", "QA", "PROD"], "Environment"
)
dbutils.widgets.dropdown(
    "order_status", "DELIVERED",
    ["DELIVERED", "SHIPPED", "PLACED", "CANCELLED"],
    "Order Status"
)
dbutils.widgets.text(
    "shipping_city", "", "Shipping City (optional)"
)

environment = dbutils.widgets.get("environment").upper()
order_status = dbutils.widgets.get("order_status").upper()
shipping_city = dbutils.widgets.get("shipping_city").strip()

print("environment  :", environment)
print("order_status :", order_status)
print("shipping_city:", shipping_city or "<ALL>")


In [0]:
EXPECTED_COUNTS = {
    "customers.csv": 12,
    "products.csv": 10,
    "orders.csv": 126,
    "order_items.csv": 259,
    "payments.csv": 126,
    "skew_sales.csv": 1000,
    "city_region_lookup.csv": 7,
    "inventory_snapshot.csv": 30,
}

for file_name, expected in EXPECTED_COUNTS.items():
    actual = (
        spark.read
        .option("header", True)
        .csv(f"{ctx.source_root}/{file_name}")
        .count()
    )
    print(f"{file_name:26s} expected={expected:4d} actual={actual:4d}")
    assert actual == expected

print("START HERE: PASSED")
